<a href="https://colab.research.google.com/github/PALAK803/seasonal-agriculture-performance-analysis/blob/main/Seasonal_Agriculture_Performance_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Seasonal Agriculture Performance Analysis
### VOIS AICTE Batch 1 — Major Project

**Objective:** Analyze agricultural data across seasons to identify patterns, trends, relationships, variations, and evidence-based recommendations.

The official problem statement asks students to explore, clean, compare, visualize, interpret, and document seasonal agricultural performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("seasonal_agriculture_performance_dataset.csv")
df.head()

## 1. Dataset Overview

- Records: **4,000**
- Variables: **28**
- Missing cells detected: **120**
- Duplicate rows: **0**
- Seasons: **Kharif, Rabi, Zaid**

The dataset contains farm, location, crop, seasonal, environmental, resource-use, production, economic, water-use and risk variables.

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())
print("\nSeason distribution:")
print(df["Season"].value_counts())

## 2. Data Cleaning

Numeric missing values are imputed with the median so that the analysis can use a complete analytical table without introducing extreme assumptions. No duplicate rows were found.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
clean = df.copy()

for col in numeric_cols:
    clean[col] = clean[col].fillna(clean[col].median())

print("Remaining missing cells:", clean.isna().sum().sum())

## 3. Season-wise Performance

Key metrics include yield, revenue, profit, water use, water efficiency, disease/pest risk and rainfall.

In [ ]:
season_summary = clean.groupby("Season").agg(
    Farms=("Farm_ID","count"),
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Production=("Production_Tonnes","mean"),
    Avg_Revenue=("Revenue_INR","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Avg_Water=("Water_Used_m3","mean"),
    Avg_Water_Eff=("Water_Efficiency_t_per_1000m3","mean"),
    Avg_Risk=("Disease_Pest_Risk_pct","mean"),
    Avg_Rainfall=("Rainfall_mm","mean")
).round(2)

season_summary

In [ ]:
season_summary[["Avg_Yield","Avg_Profit","Avg_Water_Eff"]].plot(
    kind="bar", figsize=(10,5), title="Season-wise Agricultural Performance"
)
plt.ylabel("Value (mixed scale)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Season findings

- **Kharif** has the highest average yield (**5.63 t/ha**) and average profit (**₹178,915**).
- **Rabi** averages **5.04 t/ha** yield and **₹87,689** profit.
- **Zaid** has the lowest average yield (**4.64 t/ha**) and negative average profit (**₹-24,805**).
- Average rainfall is highest in Kharif (**849.2 mm**) and lowest in Zaid (**304.6 mm**).

## 4. Crop Analysis

In [ ]:
crop_summary = clean.groupby("Crop").agg(
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Avg_Revenue=("Revenue_INR","mean"),
    Water_Efficiency=("Water_Efficiency_t_per_1000m3","mean")
).round(2).sort_values("Avg_Profit", ascending=False)

crop_summary

In [ ]:
crop_summary["Avg_Profit"].sort_values().plot(
    kind="barh", figsize=(9,5), title="Average Profit by Crop"
)
plt.xlabel("Average Profit (₹)")
plt.tight_layout()
plt.show()

### Crop findings

- **Sugarcane** has the highest average yield (**46.64 t/ha**) and highest average profit (**₹817,188**).
- **Chilli** is the second-highest crop by average profit (**₹750,878**).
- **Wheat, Rice and Maize** show negative average profit in this dataset.
- Because crops have very different production scales, crop-to-crop comparisons should be interpreted within crop context.

## 5. Crop × Season Comparison

In [ ]:
season_crop = clean.pivot_table(
    index="Crop", columns="Season",
    values="Yield_Tonnes_Ha", aggfunc="mean"
).round(2)

season_crop

In [ ]:
season_crop.plot(kind="bar", figsize=(10,5), title="Crop Yield Across Seasons")
plt.ylabel("Average Yield (tonnes/ha)")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

## 6. Irrigation Analysis

In [ ]:
irr_profit = clean.pivot_table(
    index="Irrigation_Method", columns="Season",
    values="Profit_INR", aggfunc="mean"
).round(0)

irr_profit

In [ ]:
irr_profit.T.plot(kind="bar", figsize=(10,5),
                  title="Average Profit by Irrigation Method and Season")
plt.ylabel("Average Profit (₹)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Relationships

Correlation is used as a screening tool to identify linear associations; it does not establish causation.

In [ ]:
corr_cols = [
    "Rainfall_mm","Avg_Temperature_C","Soil_Moisture_pct",
    "Fertilizer_kg_ha","Water_Used_m3",
    "Water_Efficiency_t_per_1000m3","Disease_Pest_Risk_pct",
    "Revenue_INR","Profit_INR","Yield_Tonnes_Ha"
]
corr = clean[corr_cols].corr()["Yield_Tonnes_Ha"].sort_values()
corr

In [ ]:
sample = clean[["Rainfall_mm","Yield_Tonnes_Ha"]].dropna()
r = sample.corr().iloc[0,1]

plt.figure(figsize=(9,5))
plt.scatter(sample["Rainfall_mm"], sample["Yield_Tonnes_Ha"], alpha=0.35, s=12)
plt.title(f"Rainfall vs Yield (correlation = {r:.3f})")
plt.xlabel("Rainfall (mm)")
plt.ylabel("Yield (tonnes/ha)")
plt.tight_layout()
plt.show()

# Conclusions & Recommendations

### Conclusions
1. **Kharif is the strongest overall season** on average yield and profit in the dataset.
2. **Zaid is the weakest economically**, with negative average profit.
3. **Sugarcane dominates yield and profit**, while several other crops have much lower or negative average profitability.
4. Irrigation strategy is associated with different profit outcomes across seasons; **Drip performs strongly in Kharif and Rabi**, while Zaid remains challenging.
5. Yield has a stronger linear association with **profit (r ≈ 0.488)** and water-efficiency metrics than with rainfall alone (**r ≈ 0.031**).

### Recommendations
- Prioritize evidence-based crop and season planning using historical performance.
- Evaluate efficient irrigation methods, especially where water use and profitability must be balanced.
- Investigate Zaid-season cost and productivity drivers before expanding cultivation.
- Analyze crop-specific economics rather than relying only on overall averages.
- Extend the analysis with regional comparisons and predictive models in future work.